# Change in parametrization

The parameters you chose to explore with the MCMC may differ from those required by the model. In this example, we'll see how to lift the degeneracy between inclination and scaled semimajor axis by using instead the impact parameter and the stellar density. 

In this Jupyter Notebook, I will report only the additions/modifications to the `MCMC_handson_template`


## Define the functions for parameter conversions 

First of all, you need to define the functions that will convert your MCMC parameters into the parameters required by your forward modelling code:

In [ ]:
def convert_rho_to_ars(P, rho):
    return np.power(Gsi * (d2s * d2s) * (P**2)
                    * rho * rho_Sun / (3. * np.pi), 1./3.)

def convert_b_to_i(b, e, o, a):
    o_rad  = o  / 180. * np.pi

    rho_e = (1. - e ** 2) / (1. + e * np.sin(o_rad))
    arccos_argument = b / a / rho_e
    if np.size(arccos_argument) <= 1:

        if arccos_argument > 1.:
            arccos_argument = 1
        if arccos_argument < -1.:
            arccos_argument = -1
    else:
        arccos_argument = [1. if b > 1. else b for b in arccos_argument]
        arccos_argument = [-1. if b < -1. else b for b in arccos_argument]

    return np.arccos(arccos_argument)*180./np.pi

You will need to change the starting point and the boundaries as well:

In [ ]:
theta = np.empty(12)

theta[0] = 2458956.30  #time of inferior conjunction
theta[1] = 1.4200      #orbital period
theta[2] = 0.150       #planet radius (in units of stellar radii)
theta[3] = 1.7        #STELLAR DENSITY
theta[4] = 0.5        #IMPACT PARAMETER
theta[5] = 0.50        # TESS LD coeff u1
theta[6] = 0.10        # TESS LD coeff u2
theta[7] = 1.00        # normalization factor
theta[8] = 0.0001      # jitter parameter for TESS data
theta[9] = 150.0      # RV semiamplitude in m/s
theta[10] = -38.1       # RV offset in km/s
theta[11] = 1.0      # jitter parameter for RV data

labels = ['Tc', 'P', 'Rp/Rs', 'rho_star', 'b', 'u1', 'u2', 'norm', 'TESS jitter',  'K', 'RV offset', 'RV jitter' ]

In [ ]:
boundaries = np.empty([2, len(theta)])

boundaries[:,0] = [theta[0]-0.1, theta[0]+0.1]
boundaries[:,1] = [theta[1]-0.1, theta[1]+0.1]
boundaries[:,2] = [0.0, 0.5]
boundaries[:,3] = [0.0, 5.0]  #STELLAR DENSITY
boundaries[:,4] = [0.0, 1.0] #IMPACT PARAMETER
boundaries[:,5] = [-1.00, 1.0]
boundaries[:,6] = [0.00, 1.0]
boundaries[:,7] = [0.9, 1.1]
boundaries[:,8] = [0.0, 0.01]
boundaries[:,9] = [0.0, 300.0]
boundaries[:,10] = [-38.4, -37.8]
boundaries[:,11] = [0.0, 50.0]

The forward modelling code will now accept the transformed parameters rather than the MCMC parameters

In [ ]:
tm = RoadRunnerModel('quadratic')
tm.set_data(data_lc['time'])

scaled_semimajor_axis = convert_rho_to_ars(theta[1], theta[3])
inclination = convert_b_to_i(theta[4], 0, 90., scaled_semimajor_axis)
print(scaled_semimajor_axis, inclination)
model = tm.evaluate(t0=theta[0],
                    p=theta[1], 
                    k=theta[2],  
                    a=scaled_semimajor_axis, 
                    i=inclination/180.*np.pi,
                    ldc=[theta[5], theta[6]])
....

The likelihood function must reflect these changes:

In [ ]:
def log_likelihood_photometry(theta):

    scaled_semimajor_axis = convert_rho_to_ars(theta[1], theta[3])
    inclination = convert_b_to_i(theta[4], 0, 90., scaled_semimajor_axis)

    flux = tm.evaluate(t0=theta[0],
                    p=theta[1], 
                    k=theta[2],  
                    a=scaled_semimajor_axis, 
                    i=inclination/180.*np.pi,
                    ldc=[theta[5], theta[6]])


The advantage of using the **stellar density** is that we may already have (more or less) precise knowledge of its value from an independent analysis, e.g., by combining isochrones fitting with photometry and Gaia parallax:

In [ ]:
#stellar density:
#rho_star = 1.700 \pm 0.34 rho_sun 

def log_prior(theta):
    prior = 0.00
    prior+= np.log(stats.norm.pdf(theta[3], loc=1.70, scale=0.34))
    prior+= np.log(stats.norm.pdf(theta[5], loc=0.50, scale=0.05))
    prior+= np.log(stats.norm.pdf(theta[6], loc=0.10, scale=0.05))
    return prior
print(log_prior(theta))

Remember to update the forward model's input parameters to use those derived by the MCMC sampler. Other than that, the rest of the Jupyter notebook is not affected by the reparametrization.